# Road Safety — Motorcycle Helmet Detection
## Model 1 of 2 : **RT-DETR-L**

**Task** : Detect bikers / passengers *with* and *without* helmets on roads.  
**Dataset** : [Helmet and no helmet rider detection](https://universe.roboflow.com/gw-khadatkar-and-sv-wasule/helmet-and-no-helmet-rider-detection) — 1 533 road-traffic images  
**Classes** : `With Helmet` · `Without Helmet` · `Licence Plate`  
**Model** : `rtdetr-l.pt` (Ultralytics)      

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise SystemError('❌ No GPU detected! Go to Runtime → Change runtime type → GPU')

print(gpu_info.stdout)
print('✅ GPU is available')

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()
print('\n✅ ultralytics', ultralytics.__version__)

In [ ]:
RF_API_KEY   = 'YOUR_ROBOFLOW_API_KEY_HERE'

RF_WORKSPACE = 'gw-khadatkar-and-sv-wasule'
RF_PROJECT   = 'helmet-and-no-helmet-rider-detection'
RF_VERSION   = 6

EPOCHS       = 60
IMG_SIZE     = 640
BATCH        = 8          # reduce to 4 if Colab OOM
MODEL        = 'rtdetr-l.pt'

N_PREVIEW    = 12
CONF         = 0.35

import os
from pathlib import Path

RUNS_DIR = Path('/content/runs/helmet_roadsafety')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Config ready')
print(f'  Model   : {MODEL}')
print(f'  Epochs  : {EPOCHS}  |  Batch : {BATCH}  |  Img : {IMG_SIZE}')

In [ ]:
from roboflow import Roboflow

rf      = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov8')

DATA_YAML = Path(dataset.location) / 'data.yaml'
print(f'\n✅ Dataset downloaded → {dataset.location}')
print(f'   data.yaml → {DATA_YAML}')

# Show class info
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print(f'\n   Classes ({cfg["nc"]}) : {cfg["names"]}')

In [ ]:
from ultralytics import RTDETR

model = RTDETR(MODEL)

results = model.train(
    data          = str(DATA_YAML),
    epochs        = EPOCHS,
    imgsz         = IMG_SIZE,
    batch         = BATCH,
    device        = 0,
    project       = str(RUNS_DIR),
    name          = 'rtdetr_train',
    exist_ok      = True,

    lr0           = 2e-4,
    weight_decay  = 1e-4,
    warmup_epochs = 5,
    cos_lr        = True,
    close_mosaic  = 0,
    
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    flipud        = 0.0,   
    fliplr        = 0.5,
    translate     = 0.1,
    scale         = 0.5,
)

BEST_WEIGHTS = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\nTraining complete')
print(f'   Best weights → {BEST_WEIGHTS}')

In [ ]:
import cv2, glob, math, time
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import RTDETR
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

val_dir   = Path(dataset.location) / cfg.get('val', 'valid/images')
img_paths = sorted(glob.glob(str(val_dir / '*.jpg')))[:N_PREVIEW]
if not img_paths:
    img_paths = sorted(glob.glob(str(val_dir / '*.png')))[:N_PREVIEW]
print(f'Running inference on {len(img_paths)} images …')

inf_model = RTDETR(str(BEST_WEIGHTS))
t0   = time.time()
preds = inf_model.predict(img_paths, conf=CONF, imgsz=IMG_SIZE,
                           device=0, verbose=False)
fps  = len(img_paths) / (time.time() - t0)

def get_color(name: str):
    n = name.lower()
    if 'without' in n or 'no' in n:   return (220,  50,  50)   # red   — VIOLATION
    if 'with'    in n or 'helmet' in n: return (  0, 200,  80)  # green — SAFE
    if 'licence' in n or 'plate'  in n: return ( 50, 120, 220)  # blue  — plate
    return (255, 165, 0)


frames = []
for r in preds:
    img = r.orig_img.copy()
    for box in r.boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cls_name     = r.names[int(box.cls[0])]
        conf_val     = float(box.conf[0])
        color        = get_color(cls_name)
        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
        label = f'{cls_name} {conf_val:.2f}'
        (tw,th),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(img, (x1,y1-th-5),(x1+tw+2,y1), color, -1)
        cv2.putText(img, label, (x1+1,y1-3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
    img = cv2.resize(img, (320, 240))
    frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

cols = 4
rows = math.ceil(len(frames) / cols)
fig, axes = plt.subplots(rows, cols,
                          figsize=(cols*4.2, rows*3.4),
                          facecolor='#111')
fig.suptitle(f'RT-DETR-L  ·  Motorcycle Helmet Detection  ·  {fps:.1f} FPS',
             fontsize=14, color='white', fontweight='bold', y=1.01)

for i, ax in enumerate(axes.flat):
    if i < len(frames):
        ax.imshow(frames[i])
        ax.set_title(f'img {i+1}', color='#aaa', fontsize=8)
    ax.axis('off')

# legend
import matplotlib.patches as mpatches
legend = [
    mpatches.Patch(color=(0/255,200/255,80/255),  label='With Helmet (safe)'),
    mpatches.Patch(color=(220/255,50/255,50/255), label='Without Helmet (violation)'),
    mpatches.Patch(color=(50/255,120/255,220/255),label='Licence Plate'),
]
fig.legend(handles=legend, loc='lower center', ncol=3,
           facecolor='#222', edgecolor='#444', labelcolor='white',
           fontsize=10, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
out_inf = RUNS_DIR / 'inference_grid_rtdetr.png'
plt.savefig(out_inf, dpi=130, bbox_inches='tight', facecolor='#111')
plt.show()
print(f'\n✅ Inference grid saved → {out_inf}')
print(f'   FPS : {fps:.2f}  |  Images : {len(img_paths)}')
print('\n⚠️  Review boxes above. Green = helmet on, Red = VIOLATION.')
print('   If predictions look correct, proceed to the YOLOv8 notebook.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

csv_path = RUNS_DIR / 'rtdetr_train' / 'results.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

rename = {
    'metrics/mAP50(B)':    'mAP50',
    'metrics/mAP50-95(B)': 'mAP50_95',
    'train/box_loss':      'train_box_loss',
    'val/box_loss':        'val_box_loss',
    'metrics/precision(B)':'precision',
    'metrics/recall(B)':   'recall',
}
df.rename(columns={k:v for k,v in rename.items() if k in df.columns}, inplace=True)
ep = df['epoch'] if 'epoch' in df.columns else range(len(df))

BG = '#0d0d0d';  PAN = '#1a1a1a';  TXT = '#e0e0e0'
fig = plt.figure(figsize=(16, 9), facecolor=BG)
fig.suptitle('RT-DETR-L  ·  Training Curves  ·  Motorcycle Helmet Detection',
             fontsize=15, color=TXT, fontweight='bold')
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

panels = [
    ('mAP50',          'mAP@0.5',           '#00e676', gs[0,0]),
    ('mAP50_95',       'mAP@0.5:0.95',      '#00b0ff', gs[0,1]),
    ('precision',      'Precision',          '#ff9100', gs[0,2]),
    ('recall',         'Recall',             '#ea80fc', gs[1,0]),
    ('train_box_loss', 'Box Loss (train)',   '#ff5252', gs[1,1]),
    ('val_box_loss',   'Box Loss (val)',     '#ffd740', gs[1,2]),
]

for col, title, color, pos in panels:
    ax = fig.add_subplot(pos)
    ax.set_facecolor(PAN)
    if col in df.columns:
        ax.plot(ep, df[col], color=color, linewidth=2.2)
        ax.fill_between(ep, df[col], alpha=0.15, color=color)
    ax.set_title(title, color='#ccc', fontsize=11)
    ax.tick_params(colors='#888')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(color='#2a2a2a', linewidth=0.5)
    ax.set_xlabel('Epoch', color='#666', fontsize=9)

out_curves = RUNS_DIR / 'training_curves_rtdetr.png'
plt.savefig(out_curves, dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Training curves saved → {out_curves}')

In [ ]:
import json
from ultralytics import RTDETR

val_model = RTDETR(str(BEST_WEIGHTS))
val_res   = val_model.val(imgsz=IMG_SIZE, device=0, verbose=True)

m = val_res.results_dict
payload = {
    'model':     'RT-DETR-L',
    'mAP50':     round(float(m.get('metrics/mAP50(B)',    0)), 4),
    'mAP50_95':  round(float(m.get('metrics/mAP50-95(B)', 0)), 4),
    'precision': round(float(m.get('metrics/precision(B)',0)), 4),
    'recall':    round(float(m.get('metrics/recall(B)',   0)), 4),
}

metrics_path = RUNS_DIR / 'rtdetr_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(payload, f, indent=2)

print('\n' + '='*55)
print('  RT-DETR-L  Final Validation Metrics')
print('='*55)
for k,v in payload.items():
    if k != 'model':
        print(f'  {k:<15} : {v:.4f}')
print('='*55)
print(f'\n✅ Metrics saved → {metrics_path}')
print('\n👉 Now open the YOLOv8 notebook and run it.')

In [ ]:
import shutil
zip_path = '/content/rtdetr_results'
shutil.make_archive(zip_path, 'zip', str(RUNS_DIR))
print(f'✅ Download ready: {zip_path}.zip')

from google.colab import files
files.download(zip_path + '.zip')